### Hist Gradient Boosting Classifier

In [1]:
import numpy as np, pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier as HGB
from sklearn.metrics import roc_auc_score
from data_loaders import Training_set

In [3]:
ts = Training_set()
F = ts.create_features()                                
print(F.shape, '| NaNs:', int(F.isna().sum().sum()))

(490896, 98) | NaNs: 0


In [11]:
F.head(5)

price  inflow_Langesae  inflow_Staavatn  inflow_Kjelavatn  \
starttime  step                                                              
2015-01-01 0     27.07             0.63             0.81              1.06   
           1     26.93             0.63             0.82              1.06   
           2     26.83             0.62             0.82              1.06   
           3     26.81             0.62             0.81              1.06   
           4     26.97             0.62             0.81              1.05   

                 inflow_Bordalsvatn  inflow_Foersvatn  inflow_Hyljelihyl  \
starttime  step                                                            
2015-01-01 0                   0.90              0.09               0.61   
           1                   0.89              0.09               0.60   
           2                   0.90              0.09               0.61   
           3                   0.89              0.09               0.60   
           4                   0.88              0.09               0.60   

                 inflow_Venemo  inflow_Bitdalsvatn  inflow_Songavatn  ...  \
starttime  step                                                       ...   
2015-01-01 0              0.57                1.83              8.42  ...   
           1              0.59                1.81              8.35  ...   
           2              0.59                1.79              8.25  ...   
           3              0.59                1.79              8.25  ...   
           4              0.59                1.78              8.20  ...   

                 inflow_Songavatn_wk  inflow_Totak_wk  inflow_Vaamarvatn_wk  \
starttime  step                                                               
2015-01-01 0                7.435952         5.143155              0.337738   
           1                7.435952         5.143155              0.337738   
           2                7.435952         5.143155              0.337738   
           3                7.435952         5.143155              0.337738   
           4                7.435952         5.143155              0.337738   

                 inflow_Langeidvatn_wk  inflow_Vatjern_wk  \
starttime  step                                             
2015-01-01 0                  0.547619           0.055952   
           1                  0.547619           0.055952   
           2                  0.547619           0.055952   
           3                  0.547619           0.055952   
           4                  0.547619           0.055952   

                 inflow_Vinjevatn_wk  inflow_Botnedalsvatn_wk  \
starttime  step                                                 
2015-01-01 0                7.047798                 1.703155   
           1                7.047798                 1.703155   
           2                7.047798                 1.703155   
           3                7.047798                 1.703155   
           4                7.047798                 1.703155   

                 inflow_Byrtevatn_wk  inflow_Bandak_wk  \
starttime  step                                          
2015-01-01 0                1.265179         15.793512   
           1                1.265179         15.793512   
           2                1.265179         15.793512   
           3                1.265179         15.793512   
           4                1.265179         15.793512   

                 inflow_r_Vest_Vassdraget_wk  
starttime  step                               
2015-01-01 0                        1.685476  
           1                        1.685476  
           2                        1.685476  
           3                        1.685476  
           4                        1.685476  

[5 rows x 98 columns]

In [8]:
Y = ts.create_labels

In [9]:
tr = (F.index.get_level_values(0).year <= 2021)
va = ~tr

# baseline: per-unit share of hours ON
base = np.tile(Y[tr].mean().values, (va.sum(), 1))                               
print('baseline micro AUC:', round(roc_auc_score(Y[va], base, average='micro'), 4))

pred = pd.DataFrame(index=Y.index[va], columns=Y.columns, dtype=float)
for u in Y.columns:
    m = HGB(max_iter=150, random_state=0).fit(F[tr], Y.loc[tr, u])
    pred[u] = m.predict_proba(F[va])[:, 1]
    print(f'{u:15s} AUC {roc_auc_score(Y.loc[va, u], pred[u]):.3f}')
print('HGB micro AUC:', round(roc_auc_score(Y[va], pred, average='micro'), 4))

baseline micro AUC: 0.5545
Hogga_G1        AUC 0.898
Tokke_G1        AUC 0.866
Tokke_G2        AUC 0.892
Tokke_G3        AUC 0.900
Tokke_G4        AUC 0.879
Lio_G1          AUC 0.697
Byrte_G1        AUC 0.832
Vinje_G1        AUC 0.835
Vinje_G2        AUC 0.904
Vinje_G3        AUC 0.880
Haukeli_G1      AUC 0.873
Songa_G1        AUC 0.771
Kjela_G1        AUC 0.701
Vesle Kjela_G1  AUC 0.956
HGB micro AUC: 0.8396
